<a href="https://colab.research.google.com/github/erokemwa/Blog-AI/blob/master/chat_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sqlite3
from datetime import datetime
import time
import random

# Mock LLM (replace with real API call in production)
class MockLLM:
    def generate_response(self, user_input):
        # Simulate API call delay
        time.sleep(0.5)

        # Mock different response types
        responses = {
            "hello": "Hello! How can I help you today?",
            "bye": "Goodbye! Have a great day!",
            "default": f"I understand you're saying: '{user_input}'. Can you clarify?"
        }

        return responses.get(user_input.lower(), responses["default"])

# Database manager
class DatabaseManager:
    def __init__(self, db_name="interactions.db"):
        self.conn = sqlite3.connect(db_name)
        self._create_table()

    def _create_table(self):
        self.conn.execute('''CREATE TABLE IF NOT EXISTS interactions
                            (id INTEGER PRIMARY KEY AUTOINCREMENT,
                             user_id TEXT,
                             input TEXT,
                             output TEXT,
                             timestamp DATETIME,
                             feedback INTEGER)''')
        self.conn.commit()

    def save_interaction(self, user_id, user_input, llm_output):
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        self.conn.execute('''INSERT INTO interactions
                          (user_id, input, output, timestamp)
                          VALUES (?, ?, ?, ?)''',
                          (user_id, user_input, llm_output, timestamp))
        self.conn.commit()

    def add_feedback(self, interaction_id, feedback):
        self.conn.execute('''UPDATE interactions
                          SET feedback = ?
                          WHERE id = ?''', (feedback, interaction_id))
        self.conn.commit()

    def close(self):
        self.conn.close()

# Main application
class ChatApp:
    def __init__(self):
        self.llm = MockLLM()
        self.db = DatabaseManager()
        self.current_user = "user_123"  # Mock authentication

    def process_input(self, user_input):
        # Get LLM response
        llm_output = self.llm.generate_response(user_input)

        # Store interaction
        self.db.save_interaction(self.current_user, user_input, llm_output)

        return llm_output

    def get_feedback(self):
        try:
            interaction_id = int(input("\nWas this response helpful? (1=Yes, 0=No): "))
            self.db.add_feedback(self._get_last_id(), interaction_id)
        except:
            print("Invalid feedback format")

    def _get_last_id(self):
        cursor = self.conn.execute("SELECT last_insert_rowid()")
        return cursor.fetchone()[0]

# CLI Interface
def main():
    app = ChatApp()

    print("Simple Chat App (type 'exit' to quit)")
    while True:
        user_input = input("\nYou: ")

        if user_input.lower() == 'exit':
            break

        # Process input
        response = app.process_input(user_input)
        print(f"Assistant: {response}")

        # Collect feedback
        app.get_feedback()

    app.db.close()
    print("Session ended. Goodbye!")

if __name__ == "__main__":
    main()

Simple Chat App (type 'exit' to quit)

You: hello
Assistant: Hello! How can I help you today?

Was this response helpful? (1=Yes, 0=No): 1
Invalid feedback format

You: Yes
Assistant: I understand you're saying: 'Yes'. Can you clarify?

Was this response helpful? (1=Yes, 0=No): 1=Yes
Invalid feedback format

You: Yes
Assistant: I understand you're saying: 'Yes'. Can you clarify?

Was this response helpful? (1=Yes, 0=No): Yes
Invalid feedback format


KeyboardInterrupt: Interrupted by user